In [1]:
import os
import pandas as pd
import numpy as np
import re

In [2]:
INPUT_FOLDER = "prompt_similarity_outputs"  # where your 4 CSVs are
OUTPUT_FOLDER = "prompts_sim_stats"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [3]:
FILES = [
    "reasoning_prompt_similarity.csv",
    "logical_prompt_similarity.csv",
    "qna_prompt_similarity.csv",
    "classification_prompt_similarity.csv"
]

In [4]:
for file in FILES:
    
    print(f"Processing {file}...")
    
    df = pd.read_csv(os.path.join(INPUT_FOLDER, file))
    
    # Get prompt base names (without variant)
    prompt_columns = df.columns[1:]  # skip Drift/Variant column
    
    base_prompt_map = {}
    
    for col in prompt_columns:
        base = re.sub(r'_V\d+', '', col)
        if base not in base_prompt_map:
            base_prompt_map[base] = []
        base_prompt_map[base].append(col)
    
    # ==========================================
    # 1️⃣ Mean and Std per Prompt (across variants)
    # ==========================================
    
    for base, cols in base_prompt_map.items():
        df[f"{base}_mean"] = df[cols].mean(axis=1)
        df[f"{base}_std"] = df[cols].std(axis=1)
    
    # ==========================================
    # 2️⃣ Overall Mean per Drift Level
    # ==========================================
    
    variant_cols = list(prompt_columns)
    
    df["overall_mean"] = df[variant_cols].mean(axis=1)
    df["overall_std"] = df[variant_cols].std(axis=1)
    
    # ==========================================
    # 3️⃣ Relative Drop %
    # ==========================================
    
    # Using Drift Level 1 as reference (since L0 not included)
    baseline = df.loc[df["Drift/Variant"] == 1, "overall_mean"].values[0]
    
    df["relative_drop_%"] = ((baseline - df["overall_mean"]) / baseline) * 100
    
    # Round values for readability
    df = df.round(3)
    
    # ==========================================
    # Save file
    # ==========================================
    
    save_path = os.path.join(OUTPUT_FOLDER, file.replace(".csv", "_stats.csv"))
    df.to_csv(save_path, index=False)
    
    print(f"Saved → {save_path}")

print("All files processed successfully.")

Processing reasoning_prompt_similarity.csv...
Saved → prompts_sim_stats\reasoning_prompt_similarity_stats.csv
Processing logical_prompt_similarity.csv...
Saved → prompts_sim_stats\logical_prompt_similarity_stats.csv
Processing qna_prompt_similarity.csv...
Saved → prompts_sim_stats\qna_prompt_similarity_stats.csv
Processing classification_prompt_similarity.csv...
Saved → prompts_sim_stats\classification_prompt_similarity_stats.csv
All files processed successfully.
